# Urdu OCR — Quickstart (run everything in one go)

**Before running anything: `Runtime > Change runtime type > T4 GPU`.**

This single notebook clones the repo, installs dependencies, generates a
synthetic starter dataset, fine-tunes TrOCR on it, evaluates it (prints
real CER/WER/accuracy), and launches a live demo — in that order, one
cell at a time. Run cells top to bottom.

Every cell is written to be **safe to re-run** (idempotent `cd`, no nested
clones) if your session disconnects partway through — just re-run from
Cell 1.


In [ ]:
# Cell 1 — Clone the repo (safe to re-run: pulls latest if it already
# exists, instead of silently reusing a stale checkout) and install deps.
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch.
# All of this pipeline's fixes (dependency pins included) have been pushed
# here; the default branch only has the original version. If you've since
# merged this branch into your default branch, change BRANCH below to that
# (e.g. 'main') -- but check first: an unqualified `git clone`/`git pull`
# silently reports "Already up to date" against whatever branch it's
# already tracking, which is exactly how earlier fixes here failed to
# reach a Colab session that had already cloned once.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. `!python src/train.py` in Cell 4
# runs as a fresh subprocess, so it always uses what's genuinely on disk
# -- this check mirrors that rather than this kernel's own (possibly
# stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )

# Functional smoke test, not just a version check -- and run in a genuinely
# fresh subprocess, not this notebook kernel. Pillow (and any package with a
# compiled C extension) cannot have its native .so swapped out in an
# already-running process even after a correct on-disk reinstall; the only
# way to know it will actually work is to check from a brand new process,
# which is also exactly what `!python src/train.py` etc. do in later cells.
pil_check = subprocess.run(
    ['python3', '-c',
     "from PIL import Image; im = Image.new('RGB', (32, 32), 'white'); "
     "im.save('/tmp/_pil_smoke_test.png'); "
     "Image.open('/tmp/_pil_smoke_test.png').load(); print('PIL round-trip OK')"],
    capture_output=True, text=True,
)
print(pil_check.stdout.strip())
if pil_check.returncode != 0:
    raise RuntimeError(
        'Pillow is installed but broken (fails to save/load an image even in '
        'a fresh process) -- this is not a version-pin issue, it is a '
        'corrupted install, most likely leftover mixed files on disk from '
        'earlier pip installs/reinstalls in this same Colab VM. '
        '`Runtime > Restart session` will NOT fix this (it only restarts '
        'the Python process, not the disk). You need '
        '`Runtime > Disconnect and delete runtime`, then reconnect to a '
        'brand new VM and run this notebook from Cell 1 on that clean '
        'slate.\n\nOriginal error:\n' + pil_check.stderr[-1500:]
    )

print('Setup complete -- all pinned versions correctly installed on disk and working.')


In [ ]:
# Cell 2 — Confirm GPU is attached. If this prints False, go to
# Runtime > Change runtime type > T4 GPU, then Runtime > Restart session,
# and re-run from Cell 1.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Training will be extremely slow on CPU.')


## Generate a starter dataset

The repo ships the Urdu sentence corpus and fonts but not `data/labels.csv`
itself (it's gitignored, since it's generated, not hand-written). This
renders 400 synthetic images across 3 Urdu fonts/varied backgrounds — see
`docs/ACCURACY_NOTES.md` for why. Bump `num_images` up (or add your own
real photographed images to `data/raw/` + `data/labels_real.csv`, then
merge with `synthetic.merge_labels_csv`) for a stronger model once this
first run works end to end.


In [ ]:
# Cell 3 — Generate synthetic training data.
from synthetic import generate_dataset

os.makedirs('data', exist_ok=True)
stats = generate_dataset(
    output_dir='data/raw/synthetic',
    labels_csv='data/labels.csv',
    num_images=400,
    pseudo_fraction=0.3,
    seed=42,
)
print(f'Generated {stats.total} images ({stats.from_corpus} corpus + {stats.from_pseudo} pseudo)')
print('Fonts used:', stats.fonts_used)


## Train

~15-30 minutes on a free-tier T4 for 400 images / 6 epochs. Watch for
"Epoch ... eval_cer: ..." lines in the output — CER should trend down.
If this cell's output ends with a traceback instead of "Best model saved
to: ...", **stop and paste me the error** rather than continuing to the
next cell — evaluation will fail with a confusing, unrelated-looking error
if training didn't actually finish.


In [ ]:
# Cell 4 — Fine-tune TrOCR on the generated dataset.
!python src/train.py \
    --labels-csv data/labels.csv \
    --output-dir models/urdu-trocr \
    --epochs 6 --batch-size 8

import os
assert os.path.isdir('models/urdu-trocr/final'), (
    'Training did not produce models/urdu-trocr/final -- scroll up for the '
    'real error and do not continue to Cell 5 yet.'
)
print('Training finished, model saved to models/urdu-trocr/final')


## Evaluate

This prints your actual CER / WER / exact-match accuracy, plus the 5 worst predictions — this is the real number for your README.

In [ ]:
# Cell 5 — Evaluate on the held-out test set.
!python src/evaluate.py models/urdu-trocr/final \
    --test-csv data/test.csv \
    --report-csv reports/eval_report.csv


In [ ]:
# Cell 6 — (optional) preview the full per-sample report as a table.
import pandas as pd
report = pd.read_csv('reports/eval_report.csv')
report.sort_values('cer', ascending=False).head(10)


## Try it live

Launches a Gradio interface right here in Colab with a public share link — upload one of your own Urdu images and see the prediction. Screenshot this for your README (Week 5 handout's 'screenshot your working demo').

In [ ]:
# Cell 7 — Launch a live demo (Colab-hosted, public share link).
os.environ['MODEL_PATH'] = os.path.abspath('models/urdu-trocr/final')
os.environ['GRADIO_SHARE'] = 'true'

import importlib
import app
importlib.reload(app)
app.interface.launch(share=True)


## Next steps

- **Not accurate enough yet?** The single highest-leverage next step is
  more *real* photographed/downloaded images mixed into `data/labels.csv`
  (synthetic-only data plateaus — see `docs/ACCURACY_NOTES.md`). Re-run
  from Cell 3 with a bigger `num_images`, or add real data first.
- **Publish the model** so `app.py` can load it via `MODEL_PATH` without
  Colab open — see `notebooks/SI26_Week4_Train_Evaluate.ipynb`'s
  "Save your model" section for the `push_to_hub` snippet.
- **Deploy permanently** on HuggingFace Spaces — see
  `notebooks/SI26_Week5_Gradio_Deploy.ipynb` or the README's Live Demo
  section.
